# Positions & Orders — Bybit

- **Section 3** — نمایش همه open positions و pending orders
- **Section 4** — کنسل کردن همه pending orders
- **Section 5** — بستن همه open positions با Market order

> **هشدار:** Section 4 و 5 را فقط وقتی اجرا کنید که مطمئن هستید — عمل برگشت‌ناپذیر است.

In [1]:
# SECTION 1 — Imports & config
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

try:
    from dotenv import load_dotenv
    _env = _ROOT / ".env"
    if _env.exists():
        load_dotenv(_env)
        print(f"Loaded .env from {_env}")
    else:
        print(f"No .env at {_env}")
except ImportError:
    print("python-dotenv not installed")

# ── Parameters ────────────────────────────────────────────────────────────────
CATEGORY = "linear"   # spot | linear | inverse | option

# نمادهایی که می‌خواهید بررسی شوند (خالی = همه نمادهای اکانت)
SYMBOLS = ["BTCUSDT", "XAUUSDT", "XAGUSDT"]

TESTNET = os.getenv("BYBIT_TESTNET", "false").lower() == "true"
DEMO    = os.getenv("BYBIT_DEMO",    "true" ).lower() == "true"

if DEMO and TESTNET:
    raise ValueError("Use only one of BYBIT_DEMO or BYBIT_TESTNET")

_mode = "demo" if DEMO else ("testnet" if TESTNET else "mainnet")
print(f"Endpoint: {_mode}  |  category: {CATEGORY}  |  symbols: {SYMBOLS or 'ALL'}")

Loaded .env from D:\bot\ema-h1trend-exchange\.env
Endpoint: demo  |  category: linear  |  symbols: ['BTCUSDT', 'XAUUSDT', 'XAGUSDT']


In [2]:
# SECTION 2 — Bybit client
from pybit.unified_trading import HTTP

api_key    = os.getenv("BYBIT_API_KEY",    "")
api_secret = os.getenv("BYBIT_API_SECRET", "")

client = HTTP(
    testnet=TESTNET,
    demo=DEMO,
    api_key=api_key    or None,
    api_secret=api_secret or None,
)

print(f"HTTP demo={DEMO} testnet={TESTNET}  |  API key set: {bool(api_key)}")

HTTP demo=True testnet=False  |  API key set: True


In [ ]:
# SECTION 3 — نمایش Open Positions و Pending Orders

def fetch_positions(symbols: list[str]) -> pd.DataFrame:
    rows = []
    targets = symbols if symbols else [None]
    for sym in targets:
        kwargs = dict(category=CATEGORY, settleCoin="USDT")
        if sym:
            kwargs["symbol"] = sym
        resp = client.get_positions(**kwargs)
        if resp.get("retCode") != 0:
            print(f"  WARN positions {sym}: {resp.get('retMsg')}")
            continue
        for item in (resp.get("result") or {}).get("list") or []:
            if float(item.get("size", 0) or 0) != 0:
                rows.append(item)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    num_cols = ["size", "avgPrice", "markPrice", "unrealisedPnl",
                "cumRealisedPnl", "stopLoss", "takeProfit", "liqPrice", "leverage"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    keep = [c for c in ["symbol", "side", "size", "avgPrice", "markPrice",
                         "unrealisedPnl", "stopLoss", "takeProfit",
                         "liqPrice", "leverage", "positionIdx"] if c in df.columns]
    return df[keep].reset_index(drop=True)


# فقط اردرهایی که واقعاً باز هستند — Filled / Deactivated نمایش داده نمی‌شوند
_ACTIVE_STATUSES = {"New", "PartiallyFilled", "Untriggered"}

def fetch_orders(symbols: list[str]) -> pd.DataFrame:
    rows = []
    targets = symbols if symbols else [None]
    for sym in targets:
        kwargs = dict(category=CATEGORY, openOnly=0, limit=50)
        if sym:
            kwargs["symbol"] = sym
        resp = client.get_open_orders(**kwargs)
        if resp.get("retCode") != 0:
            print(f"  WARN orders {sym}: {resp.get('retMsg')}")
            continue
        rows.extend((resp.get("result") or {}).get("list") or [])
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    if "orderStatus" in df.columns:
        df = df[df["orderStatus"].isin(_ACTIVE_STATUSES)].copy()
    if df.empty:
        return pd.DataFrame()
    num_cols = ["price", "qty", "leavesQty", "stopLoss", "takeProfit",
                "triggerPrice", "avgPrice"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    keep = [c for c in ["symbol", "side", "orderType", "qty", "price",
                         "triggerPrice", "stopLoss", "takeProfit",
                         "orderStatus", "orderId", "createdTime"] if c in df.columns]
    df = df[keep].copy()
    if "createdTime" in df.columns:
        df["createdTime"] = pd.to_datetime(
            pd.to_numeric(df["createdTime"], errors="coerce"), unit="ms", utc=True
        ).dt.strftime("%Y-%m-%d %H:%M")
    return df.reset_index(drop=True)


pos_df = fetch_positions(SYMBOLS)
ord_df = fetch_orders(SYMBOLS)

print(f"\n{'═'*60}")
print(f"  OPEN POSITIONS   ({len(pos_df)} rows)")
print(f"{'═'*60}")
if pos_df.empty:
    print("  -- هیچ پوزیشن بازی وجود ندارد --")
else:
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 220):
        display(pos_df)
    total_upnl = pos_df["unrealisedPnl"].sum() if "unrealisedPnl" in pos_df.columns else 0
    print(f"  Total unrealised PnL: {total_upnl:+,.4f} USDT")

print(f"\n{'═'*60}")
print(f"  PENDING ORDERS   ({len(ord_df)} rows)  [فقط New / PartiallyFilled / Untriggered]")
print(f"{'═'*60}")
if ord_df.empty:
    print("  -- هیچ اردر فعالی وجود ندارد --")
else:
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 220):
        display(ord_df)

In [4]:
# SECTION 4 — کنسل کردن همه Pending Orders
# ⚠️  این سلول همه اردرهای فعال را کنسل می‌کند.
#     قبل از اجرا مطمئن شوید.

CONFIRM_CANCEL = True   # برای کنسل کردن اردرها True بگذارید

if not CONFIRM_CANCEL:
    print("CONFIRM_CANCEL = False  →  هیچ اردری کنسل نشد.")
elif ord_df.empty:
    print("هیچ اردر فعالی وجود ندارد.")
else:
    targets = SYMBOLS if SYMBOLS else list(ord_df["symbol"].unique())
    total_cancelled = 0
    for sym in targets:
        resp = client.cancel_all_orders(category=CATEGORY, symbol=sym)
        code = resp.get("retCode")
        cancelled = (resp.get("result") or {}).get("list") or []
        if code == 0:
            total_cancelled += len(cancelled)
            print(f"  {sym}  →  {len(cancelled)} اردر کنسل شد")
        else:
            print(f"  {sym}  →  خطا: {resp.get('retMsg')}")
    print(f"\nجمع: {total_cancelled} اردر کنسل شد.")

  BTCUSDT  →  0 اردر کنسل شد
  XAUUSDT  →  0 اردر کنسل شد
  XAGUSDT  →  0 اردر کنسل شد

جمع: 0 اردر کنسل شد.


In [5]:
# SECTION 5 — بستن همه Open Positions با Market Order
# ⚠️  این سلول همه پوزیشن‌های باز را با قیمت بازار می‌بندد.
#     قبل از اجرا مطمئن شوید.

CONFIRM_CLOSE = True    # برای بستن پوزیشن‌ها True بگذارید

if not CONFIRM_CLOSE:
    print("CONFIRM_CLOSE = False  →  هیچ پوزیشنی بسته نشد.")
elif pos_df.empty:
    print("هیچ پوزیشن بازی وجود ندارد.")
else:
    total_closed = 0
    for _, row in pos_df.iterrows():
        sym  = row["symbol"]
        size = abs(float(row["size"]))
        side = row["side"]            # "Buy" یا "Sell" از Bybit
        close_side = "Sell" if side == "Buy" else "Buy"
        pos_idx = int(row.get("positionIdx", 0))

        resp = client.place_order(
            category=CATEGORY,
            symbol=sym,
            side=close_side,
            orderType="Market",
            qty=str(size),
            reduceOnly=True,
            positionIdx=pos_idx,
        )
        code = resp.get("retCode")
        oid  = (resp.get("result") or {}).get("orderId", "")
        if code == 0:
            total_closed += 1
            print(f"  {sym}  side={side}  size={size}  →  بسته شد  orderId={oid}")
        else:
            print(f"  {sym}  side={side}  size={size}  →  خطا: {resp.get('retMsg')}")
    print(f"\nجمع: {total_closed} پوزیشن بسته شد.")

هیچ پوزیشن بازی وجود ندارد.
